In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.ma.extras import apply_over_axes
from sklearn.metrics import r2_score
import joblib

plt.style.use("seaborn-v0_8")

In [49]:
apartments = pd.read_csv("../data/rentals.csv")
apartments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   price_numeric               305 non-null    float64
 1   municipality                305 non-null    object 
 2   condition                   305 non-null    object 
 3   rooms                       305 non-null    float64
 4   square_m2                   305 non-null    float64
 5   equipment                   305 non-null    object 
 6   level                       305 non-null    int64  
 7   heating                     305 non-null    object 
 8   price_per_m2                305 non-null    float64
 9   geometry                    305 non-null    object 
 10  closest_hospital_m          305 non-null    float64
 11  closest_clinic_m            305 non-null    float64
 12  closest_pharmacy_m          305 non-null    float64
 13  closest_school_m            305 non

In [50]:
from sklearn.model_selection import train_test_split, cross_val_score

X = apartments.drop(labels=["price_numeric", "price_per_m2" ], axis="columns")
y = apartments["price_numeric"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=apartments['municipality'])

In [51]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = X.select_dtypes( include=np.number ).columns

non_numeric_CT = ColumnTransformer( transformers=[
  ('cat_OHE', OneHotEncoder( handle_unknown='ignore' ), [ "condition", "equipment", "heating" ]),
  ('numeric_transformer', "passthrough", numeric_features)
], remainder="drop" )

numeric_CT = ColumnTransformer( transformers=[
  ('cat_OHE', OneHotEncoder( handle_unknown='ignore' ), [ "condition", "equipment", "heating" ]),
  ('numeric_transformer', StandardScaler( ), numeric_features)
], remainder="drop" )

In [52]:
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

model_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ('feature_selector', "passthrough"),
  ("model", LinearRegression(n_jobs=-1))
])

In [ ]:
param_grid_RF = [
  {
    "model" : [RandomForestRegressor()],
    "model__n_estimators" : [50, 100, 200, 500],
    "model__max_depth" : [None, 2, 3, 5, 10, 20, 30],
    "model__min_samples_leaf" : [1, 5, 10, 20],
    "model__min_samples_split" : [2, 10, 20],
    "model__max_features" : [None, "sqrt", "log2"]
  },
  {
    "model" : [TransformedTargetRegressor(regressor=RandomForestRegressor(), func=np.log, inverse_func=np.exp)],
    "model__regressor__n_estimators" : [50, 100, 200, 500],
    "model__regressor__max_depth" : [None, 2, 3, 5, 10, 20, 30],
    "model__regressor__min_samples_leaf" : [1, 5, 10, 20],
    "model__regressor__min_samples_split" : [2, 10, 20],
    "model__regressor__max_features" : [None, "sqrt", "log2"]
  }
]

grid_search_RF = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_RF,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_RF.fit(X_train, y_train)

Fitting 5 folds for each of 2016 candidates, totalling 10080 fits


In [ ]:
param_grid_LinearRegression = [
  {
    "model" : [TransformedTargetRegressor(
      regressor=LinearRegression(n_jobs=-1),
      func=np.log1p,
      inverse_func=np.expm1
    ), LinearRegression(n_jobs=-1) ],
    "feature_selector" : [SelectKBest(score_func=f_regression)],
    "feature_selector__k" : [2,3,5,10,20,30, "all"]
  },
]

grid_search_LinearRegression = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_LinearRegression,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_LinearRegression.fit(X_train, y_train)

In [ ]:
param_grid_knn = [
  {
    'model' : [KNeighborsRegressor(n_jobs=-1)],
    'model__n_neighbors': [1, 3, 5, 10, 20, 30, 50, 100],
    'model__weights': ['uniform', 'distance'],
    'model__algorithm' : ['ball_tree', 'kd_tree', 'brute', 'auto'],
    'model__p' : [1, 2]
  },
  {
    'model' : [TransformedTargetRegressor(
      regressor=KNeighborsRegressor(n_jobs=-1),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],
    'model__regressor__n_neighbors': [1, 3, 5, 10, 20, 30, 50, 100],
    'model__regressor__weights': ['uniform', 'distance'],
    'model__regressor__algorithm' : ['ball_tree', 'kd_tree', 'brute', 'auto'],
    'model__regressor__p' : [1, 2]
  }
]

grid_search_knn = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_knn,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_knn.fit(X_train, y_train)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

param_grid_gb = [
  {
    "model": [HistGradientBoostingRegressor()],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter": [200, 400],
  },
  {
    "model": [TransformedTargetRegressor(
      regressor=HistGradientBoostingRegressor(),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],
    "model__regressor__max_depth": [3, 5, 7],
    "model__regressor__learning_rate": [0.05, 0.1],
    "model__regressor__max_iter": [200, 400],
  },
]

grid_search_gb = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_gb,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_gb.fit(X_train, y_train)

In [ ]:
param_grid_SVR = [
  {
    'model' : [SVR()],
    'model__C': [0.1, 1, 10, 100],
    'model__epsilon': [0.01, 0.1, 0.2, 0.5],
    'model__gamma': ["scale", 0.01, 0.1, 1],
    'model__kernel': ['rbf'],
  },
  {
    'model' : [SVR(kernel='linear')],
    'model__C': [0.1, 1, 10, 100],
    'model__epsilon': [0.01, 0.1, 0.2],
  },
  {
    'model' : [TransformedTargetRegressor(
      regressor=SVR(),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],
    'model__regressor__C': [0.1, 1, 10, 100],
    'model__regressor__epsilon': [0.01, 0.1, 0.5],
    'model__regressor__gamma': ["scale", 0.01, 0.1, 1],
    'model__regressor__kernel': ['rbf'],
  },
  {
    'model' : [TransformedTargetRegressor(
      regressor=SVR(kernel='linear'),
      func=np.log1p,
      inverse_func=np.expm1
    ) ],

    'model__regressor__C': [0.1, 1, 10, 100],
    'model__regressor__epsilon': [0.01, 0.1, 0.2],
  }
]

grid_search_SVR = GridSearchCV(
  estimator = model_pipeline,
  param_grid = param_grid_SVR,
  n_jobs = -1,
  cv = 5,
  verbose = 1,
  scoring = 'r2',
)

grid_search_SVR.fit(X_train, y_train)

In [ ]:
best_models = {
  'RandomForest' : grid_search_RF.best_estimator_,
  'LinearRegression' : grid_search_LinearRegression.best_estimator_,
  'KNeighborsRegressor' : grid_search_knn.best_estimator_,
  'SVR' : grid_search_SVR.best_estimator_,
  'GradientBoostingRegressor' : grid_search_gb.best_estimator_,
}

from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

report_df = pd.DataFrame(columns=['CV_R2', 'Test_R2', 'CV_NMAE', 'Test_NMAE', 'Baseline_NMAE', 'Model'])

for model in best_models.keys():
    best_model = best_models[model]
    y_pred = best_model.predict(X_test)

    report_entry = {
      'Model' : model,
      'CV_R2' : cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2').mean(),
      'Test_R2' : r2_score(y_test, y_pred),
      'CV_NMAE' : cross_val_score(best_model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean(),
      'Test_NMAE' : -mean_absolute_error(y_test, y_pred),
      'Baseline_NMAE' : -mean_absolute_error(y_test, np.full(len(y_test), y_test.mean())),
    }

    if len(report_entry) > 0:
      report_df = pd.concat([report_df, pd.DataFrame([report_entry])], ignore_index=True)
    else:
      report_df = pd.DataFrame([report_df])

report_df.sort_values(by=['Test_R2', 'CV_R2', 'Test_NMAE', 'CV_NMAE'], ascending=False, inplace=True)
report_df.to_latex(buf='../ProjectReport/Tables/score_rents_report_table_with.tex',index=False, column_format='|l|c|c|c|c|c|', float_format="%.3f")

In [ ]:
report_df.sort_values(by=['Test_R2', 'CV_R2', 'Test_NMAE', 'CV_NMAE'], ascending=False)

In [ ]:
name_of_the_best_model = report_df.nlargest(n = 1, columns=['Test_R2', 'CV_R2', 'Test_NMAE', 'CV_NMAE'])['Model'].values[0]
name_of_the_best_model

In [ ]:
import joblib

model_to_save = best_models[name_of_the_best_model] # Pick a model to save
model_to_save.fit(X, y)

joblib.dump(model_to_save, "../models/rentals_predict_ML.joblib")

In [ ]:
loaded = joblib.load("../models/rentals_predict_ML.joblib") # Check if the model saved is the right one

np.testing.assert_allclose(
    model_to_save.predict(X_test[:10]),
    loaded.predict(X_test[:10])
)